In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from scipy.stats import entropy

# ========== 1. Load Dataset ==========
df = pd.read_csv("customer_churn_dataset.csv")

# Ensure output folders exist
os.makedirs("plots", exist_ok=True)
os.makedirs("feature_json", exist_ok=True)

# ========== 2. Feature Type Detection ==========
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
datetime_cols = df.select_dtypes(include=['datetime64[ns]', 'datetime64']).columns.tolist()

# Handle low-cardinality numeric columns as categorical
for col in num_cols.copy():
    if df[col].nunique() < 20:
        cat_cols.append(col)
        num_cols.remove(col)

print("Numerical Columns:", num_cols)
print("Categorical Columns:", cat_cols)
print("Datetime Columns:", datetime_cols)

# Optional: detect target column
target_col = "Churn" if "Churn" in df.columns else None


Numerical Columns: ['tenure_months', 'monthly_usage_hours']
Categorical Columns: ['has_multiple_devices', 'customer_support_calls', 'payment_failures', 'is_premium_plan', 'churn']
Datetime Columns: []


In [2]:

# ========== 3. Helper Functions ==========
def calc_entropy(series):
    probs = series.value_counts(normalize=True)
    return float(entropy(probs, base=2))

def detect_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5*iqr
    upper = q3 + 1.5*iqr
    outliers = series[(series < lower) | (series > upper)]
    return len(outliers), len(outliers)/len(series)*100

def process_numerical(col):
    s = df[col].dropna()
    outlier_count, outlier_pct = detect_outliers(s)
    stats = {
        "count": int(s.count()),
        "missing_count": int(df[col].isna().sum()),
        "missing_pct": float(df[col].isna().mean()*100),
        "mean": float(s.mean()),
        "median": float(s.median()),
        "std": float(s.std()),
        "min": float(s.min()),
        "max": float(s.max()),
        "q1": float(s.quantile(0.25)),
        "q3": float(s.quantile(0.75)),
        "iqr": float(s.quantile(0.75)-s.quantile(0.25)),
        "skewness": float(s.skew()),
        "kurtosis": float(s.kurt()),
        "outlier_count": int(outlier_count),
        "outlier_pct": float(outlier_pct),
        "unique_values": int(s.nunique())
    }

    # Distribution for LLM
    hist_counts, hist_bins = np.histogram(s, bins=10)
    stats["distribution"] = {
        "bins": hist_bins.tolist(),
        "counts": hist_counts.tolist()
    }

    # Correlation with other numeric features
    correlations = {}
    for other in num_cols:
        if other != col:
            correlations[other] = float(df[col].corr(df[other]))
    if target_col and col != target_col and target_col in df.columns:
        correlations[target_col] = float(df[col].corr(df[target_col]))
    stats["correlations"] = correlations

    # Plots
    plt.figure(figsize=(6,4))
    sns.histplot(s, kde=True, bins=30)
    plt.title(f"Distribution of {col}")
    plt.savefig(f"plots/{col}_hist.png")
    plt.close()

    plt.figure(figsize=(6,4))
    sns.boxplot(x=s)
    plt.title(f"Boxplot of {col}")
    plt.savefig(f"plots/{col}_box.png")
    plt.close()

    return stats

def process_categorical(col):
    s = df[col].dropna()
    value_counts = s.value_counts()
    percentages = s.value_counts(normalize=True)*100

    stats = {
        "count": int(s.count()),
        "missing_count": int(df[col].isna().sum()),
        "missing_pct": float(df[col].isna().mean()*100),
        "unique_values": int(s.nunique()),
        "top_categories": value_counts.head(5).to_dict(),
        "percentages": percentages.head(5).to_dict(),
        "entropy": calc_entropy(s)
    }

    # Plots
    plt.figure(figsize=(6,4))
    value_counts.head(10).plot(kind='bar')
    plt.title(f"Top categories in {col}")
    plt.ylabel("Count")
    plt.savefig(f"plots/{col}_bar.png")
    plt.close()

    plt.figure(figsize=(6,6))
    value_counts.head(5).plot(kind='pie', autopct='%1.1f%%')
    plt.title(f"Distribution of {col}")
    plt.ylabel("")
    plt.savefig(f"plots/{col}_pie.png")
    plt.close()

    return stats

def process_datetime(col):
    s = df[col].dropna()
    stats = {
        "count": int(s.count()),
        "missing_count": int(df[col].isna().sum()),
        "missing_pct": float(df[col].isna().mean()*100),
        "min": str(s.min()),
        "max": str(s.max()),
        "unique_values": int(s.nunique())
    }

    # Optional trend plot
    try:
        df_sorted = df[[col]].copy()
        df_sorted['count'] = 1
        df_trend = df_sorted.groupby(col).count().cumsum()
        plt.figure(figsize=(8,4))
        plt.plot(df_trend.index, df_trend['count'])
        plt.title(f"Cumulative Trend of {col}")
        plt.xlabel(col)
        plt.ylabel("Cumulative Count")
        plt.savefig(f"plots/{col}_trend.png")
        plt.close()
    except:
        pass

    return stats

# ========== 4. Process All Features ==========
feature_insights = {}

for col in num_cols:
    feature_insights[col] = process_numerical(col)

for col in cat_cols:
    feature_insights[col] = process_categorical(col)

for col in datetime_cols:
    feature_insights[col] = process_datetime(col)

# Save all insights in one JSON
with open("feature_json/all_features.json", "w") as f:
    json.dump(feature_insights, f, indent=4)

print("✅ Plots & JSON files generated successfully!")


✅ Plots & JSON files generated successfully!
